In [ ]:
# Installing MissForst
%pip install missforest

In [1]:
# Importing Librarires
import warnings
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from missforest import MissForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

c:\ProgramData\anaconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(
C:\Users\Mark Maged\AppData\Local\Temp\ipykernel_15020\1774523943.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install data-profiling via `pip install data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
# Loading the dataset
df = spark.table("workspace.default.default_of_credit_card_clients").toPandas()
print(df.head())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_1  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month  
0         0         0   

In [3]:
# Setting target variable
y = df['default payment next month']
x = df.drop(['default payment next month', 'ID'], axis=1)

#train/test split with a ratio of 85/15
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.15, random_state= 31, stratify= y)

In [4]:
# There are 43 duplicated entries so they will be dropped
print(f'Number of duplicated entries: {x_train.duplicated().sum()}')
duplicates_mask = ~x_train.duplicated()
x_train = x_train[duplicates_mask].reset_index(drop=True)
y_train = y_train[duplicates_mask].reset_index(drop=True)

Number of duplicated entries: 43


In [5]:
# According to the documentation of the dataset
# Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).
# Marriage(1 = married; 2 = single; 3 = others).
# But there are a few observations that are not documented, and therefore will be considerd as Nulls
# We will use MissForest to figure out their real value
undocumented = {
    'EDUCATION': {0: np.nan, 5: np.nan, 6: np.nan},
    'MARRIAGE': {0: np.nan}
    }
x_train.replace(undocumented, inplace= True)
x_test.replace(undocumented, inplace= True)

In [6]:
# MissForest imputation
cat_vars = ['SEX', 'EDUCATION', 'MARRIAGE']

missforest_imputer = MissForest(clf= RandomForestClassifier(n_estimators=100, max_depth= 10, random_state= 67), categorical= cat_vars)

x_train = pd.DataFrame(missforest_imputer.fit_transform(x_train), columns=x.columns)
x_test = pd.DataFrame(missforest_imputer.transform(x_test), columns=x.columns)

100%|██████████| 5/5 [00:00<00:00, 35.77it/s]


In [7]:
# Verify the imputation worked
print(x_train['EDUCATION'].value_counts())
print(x_train['MARRIAGE'].value_counts())

# Check no NaNs remain
print(f'NANs check: {x_train[['EDUCATION', 'MARRIAGE']].isna().sum()}')

EDUCATION
2.0    12127
1.0     9033
3.0     4195
4.0      102
Name: count, dtype: int64
MARRIAGE
2.0    13578
1.0    11589
3.0      290
Name: count, dtype: int64
NANs check: EDUCATION    0
MARRIAGE     0
dtype: int64


In [ ]:
# Preparing dataframes for databricks
spark.createDataFrame(x_train).write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.credit_x_train")
spark.createDataFrame(x_test).write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.credit_x_test")
spark.createDataFrame(y_train.to_frame()).write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.credit_y_train")
spark.createDataFrame(y_test.to_frame()).write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.credit_y_test")